# S³DT — Stage A: Independent Semantic-Claim Projection Validation

**Purpose:** Validate the accuracy of Django source-code claim projection *without modifying or rerunning the existing S³DT detector*.

**Input:** `TDSC_Broader_Quantitative_Audit(2).zip` (or an equivalent complete audit ZIP containing `system_sic_run/system_sic_claim_inventory.csv`, source provenance, and `pinned_source_cache/`).

**Design:** The notebook extracts all source-backed `code_behavior` claims, excludes the derived `risk_stage` and `activity` records, and deduplicates identical function/template excerpts across repeated historical checkpoints. It produces a **blind human annotation packet** (source code only, no S³DT predictions or incident outcome labels). Only after independently completed annotations are returned does it compare claims and calculate precision, recall, F1, exact confidence intervals, and source-line provenance alignment.

This Stage A analysis is **independent of the Django case-study narrative**. You can report its validation findings in the *evaluation* section and put protocols/annotation detail in supplementary material or GitHub.

**Scientific boundaries:** This is a focused validation of predefined code-behaviour claim extraction for nine Django subjects, *not* a general measure of LLM extraction from all lifecycle artifacts. A single GitHub project also limits external validity.

## 1. Validation implementation

The following cell contains the complete audit and scoring implementation. It reads the **existing** Django ZIP without altering existing SIC values, drift equations, graphs, or detector results. It validates pinned source hashes and only unblinds predictions during Stage A scoring, after reference annotations have been supplied.

In [ ]:
"""S³DT Stage A: Django source-claim projection validation (independent annotation).

Produces a BLIND source-code annotation packet from existing pinned Django histories.
Only after independent human annotation does it compare source-backed S3DT claims.
This intentionally does NOT change S3DT or present automatic labels as gold.
"""
from __future__ import annotations
import ast
import csv
import hashlib
import io
import json
import re
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

SOURCE_ZIP_CLAIMS = "system_sic_run/system_sic_claim_inventory.csv"
SOURCE_ZIP_PROVENANCE = "system_sic_run/system_sic_source_provenance.csv"
BASE_DIR = "pinned_source_cache/django_django"
SUBJECT_FILE = {
    "InlineModelAdmin.has_add_permission": "django/contrib/admin/options.py",
    "InlineModelAdmin.has_change_permission": "django/contrib/admin/options.py",
    "InlineModelAdmin.has_delete_permission": "django/contrib/admin/options.py",
    "AdminURLFieldWidget.get_context": "django/contrib/admin/widgets.py",
    "admin/widgets/url.html": "django/contrib/admin/templates/admin/widgets/url.html",
    "Storage.get_available_name": "django/core/files/storage.py",
    "Storage.generate_filename": "django/core/files/storage.py",
    "UploadedFile._set_name": "django/core/files/uploadedfile.py",
    "TemplateDetailView.get_context_data": "django/contrib/admindocs/views.py",
}
# The options describe the permitted *independent reference ontology*, not the detector's predictions.
# UNRESOLVED is deliberately not included as a possible gold fact for reporting performance.
REFERENCE_OPTIONS = {
    "InlineModelAdmin.has_add_permission": ["view_permission_controls_mutation", "change_permission_controls_mutation", "other_guard", "NONE", "OTHER", "UNRESOLVED"],
    "InlineModelAdmin.has_change_permission": ["view_permission_controls_mutation", "change_permission_controls_mutation", "other_guard", "NONE", "OTHER", "UNRESOLVED"],
    "InlineModelAdmin.has_delete_permission": ["view_permission_controls_mutation", "change_permission_controls_mutation", "other_guard", "NONE", "OTHER", "UNRESOLVED"],
    "AdminURLFieldWidget.get_context": ["scheme_validation_present", "scheme_validation_not_identified", "NONE", "OTHER", "UNRESOLVED"],
    "admin/widgets/url.html": ["url_valid_gate", "url_valid_gate_missing", "NONE", "OTHER", "UNRESOLVED"],
    "Storage.get_available_name": ["path_validation_present", "path_validation_not_identified", "NONE", "OTHER", "UNRESOLVED"],
    "Storage.generate_filename": ["path_validation_present", "path_validation_not_identified", "NONE", "OTHER", "UNRESOLVED"],
    "UploadedFile._set_name": ["path_validation_present", "path_validation_not_identified", "NONE", "OTHER", "UNRESOLVED"],
    "TemplateDetailView.get_context_data": ["safe_join_path", "direct_untrusted_path_join", "NONE", "OTHER", "UNRESOLVED"],
}
POLICY_AREAS = {
    "DJANGO-AUTHZ-001": "Admin mutation authorization: distinguish view-only permissions from modification guards.",
    "DJANGO-URL-001": "Admin URL output: examine whether URLs are checked before being used as clickable links.",
    "DJANGO-UPLOAD-001": "Uploaded filename and storage path validation: identify checks for unsafe paths.",
    "DJANGO-TEMPLATE-001": "Admin documentation template paths: examine safety of user-controlled path resolution.",
}


def _sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _read_csv(z: zipfile.ZipFile, path: str):
    with z.open(path) as fd:
        return list(csv.DictReader(io.TextIOWrapper(fd, encoding="utf-8-sig", newline="")))


def _write_csv(path: Path, rows, columns):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as fd:
        writer = csv.DictWriter(fd, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({c: row.get(c, "") for c in columns})


def _extract_excerpt(source: str, subject: str):
    """Return exact AST function source with original source-file line numbers."""
    lines = source.splitlines()
    if subject.endswith(".html"):
        return 1, len(lines), "\n".join(lines)
    cls_name, function_name = subject.split(".", 1)
    tree = ast.parse(source)
    matches = [m for cls in ast.walk(tree)
               if isinstance(cls, ast.ClassDef) and cls.name == cls_name
               for m in cls.body
               if isinstance(m, (ast.FunctionDef, ast.AsyncFunctionDef)) and m.name == function_name]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one {subject} definition; found {len(matches)}")
    m = matches[0]
    return m.lineno, m.end_lineno, "\n".join(lines[m.lineno - 1:m.end_lineno])


def _strict_source_name(z, commit, path):
    result = f"{BASE_DIR}/{commit}/{path}"
    if result not in z.namelist():
        raise FileNotFoundError(f"Pinned source missing: {result}")
    return result


def build_packet(audit_zip: str | Path, out: str | Path):
    """Stage 1: create a packet without predicted objects or vulnerability/fix labels."""
    audit_zip = Path(audit_zip)
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    packet = out / "BLIND_ANNOTATION_PACKET"
    packet.mkdir(exist_ok=True)
    excerpts = packet / "source_excerpts"
    excerpts.mkdir(exist_ok=True)
    records = {}
    occurrences = defaultdict(list)
    provenance_errors = []
    with zipfile.ZipFile(audit_zip) as z:
        claims = _read_csv(z, SOURCE_ZIP_CLAIMS)
        prov = _read_csv(z, SOURCE_ZIP_PROVENANCE)
        prov_by = {(x["case_id"], x["commit"], x["path"]): x for x in prov if x["status"] == "pinned_source"}
        code = [x for x in claims if x["predicate"] == "code_behavior"]
        other = [x for x in claims if x["predicate"] != "code_behavior"]
        for row in code:
            subj = row["subject"]
            if subj not in SUBJECT_FILE:
                raise ValueError(f"Unexpected subject. Add an explicit source extractor before scoring: {subj}")
            path = SUBJECT_FILE[subj]
            entry = _strict_source_name(z, row["commit"], path)
            data = z.read(entry)
            pr = prov_by.get((row["case_id"], row["commit"], path))
            if pr is None or _sha256(data) != pr["sha256"]:
                provenance_errors.append(entry)
            start, end, excerpt = _extract_excerpt(data.decode("utf-8"), subj)
            key = _sha256((subj + "\0" + excerpt).encode("utf-8"))
            uid = "SRC-" + key[:12]
            if uid in records and records[uid]["unique_key"] != key:
                raise ValueError("Unit-ID collision")
            rowinfo = {
                "unit_id": uid, "unique_key": key, "subject": subj,
                "source_path": path, "policy_id": row["policy_id"],
                "predicate": row["predicate"], "predicted_object": row["object"],
                "evidence": row["evidence"], "line_start": start, "line_end": end,
                "excerpt": excerpt,
            }
            if uid in records and records[uid]["predicted_object"] != row["object"]:
                raise ValueError(f"Same code excerpt received different predictions: {uid}")
            records.setdefault(uid, rowinfo)
            occurrences[uid].append({"case_id": row["case_id"], "checkpoint": row["checkpoint"],
                                    "commit": row["commit"], "evidence": row["evidence"]})
        if provenance_errors:
            raise ValueError(f"Pinned-source SHA256 mismatch or missing provenance: {provenance_errors[:4]}")

    # Annotator-facing views exclude predictions, commits, case IDs, and vulnerability/fix roles.
    gold_columns = ["unit_id", "subject", "source_excerpt_file", "policy_area", "gold_object",
                    "gold_support_lines", "gold_notes", "annotator_id"]
    gold = []
    for uid, r in sorted(records.items(), key=lambda kv: (kv[1]["policy_id"], kv[1]["subject"], kv[0])):
        excerpt_name = f"source_excerpts/{uid}.txt"
        line_numbers = range(r["line_start"], r["line_end"] + 1)
        numbered = "\n".join(f"{ln:>5} | {txt}" for ln, txt in zip(line_numbers, r["excerpt"].splitlines()))
        (packet / excerpt_name).write_text(
            f"SOURCE UNIT {uid}\nFile: {r['source_path']}\nFunction: {r['subject']}\n"
            f"Policy area: {POLICY_AREAS[r['policy_id']]}\n"
            f"Source lines: {r['line_start']}–{r['line_end']}\n\n{numbered}\n", encoding="utf-8")
        gold.append({"unit_id": uid, "subject": r["subject"], "source_excerpt_file": excerpt_name,
                     "policy_area": r["policy_id"], "gold_object": "", "gold_support_lines": "",
                     "gold_notes": "", "annotator_id": ""})
    _write_csv(packet / "gold_annotations.csv", gold, gold_columns)
    _write_csv(packet / "additional_security_claims.csv", [],
               ["unit_id", "subject", "predicate", "object", "gold_support_lines", "gold_notes", "annotator_id"])
    opts = []
    for subject, possible in REFERENCE_OPTIONS.items():
        for obj in possible:
            opts.append({"subject": subject, "allowed_gold_object": obj})
    _write_csv(packet / "REFERENCE_ONTOLOGY.csv", opts, ["subject", "allowed_gold_object"])
    (packet / "ANNOTATOR_INSTRUCTIONS.md").write_text("""# Blind annotation protocol — Django S³DT projection validation

**Purpose:** Independently describe what historical Django code excerpts support.

1. Give this folder to an annotator who has **not** inspected the S³DT predictions, case labels, or detector output. If possible, use a second annotator and adjudicate disagreements.
2. Read all `source_excerpts/SRC-*.txt` files in their entirety. Work only from the source and the policy-area description. Do **not** search for the historical fix, CVE, detection outcome, or comparison file.
3. For each `gold_annotations.csv` row, identify the security-relevant behaviour of the stated `subject`. Enter **one** `gold_object` from `REFERENCE_ONTOLOGY.csv`. Use `NONE` if the excerpt does not support a claim for that subject, `OTHER` if the behaviour is material but absent from the ontology (explain in `gold_notes`), or `UNRESOLVED` if the excerpt is insufficient or requires external call-chain context. Do **not** infer that a missing check proves a vulnerability.
4. Record `gold_support_lines` as original file lines (examples: `112-119` or `112,116,120`). Use actual numbered lines in the excerpt, not arbitrary new line numbers. Explain uncertainty in `gold_notes`.
5. Add further security-relevant claims that this one-subject-per-unit protocol misses to `additional_security_claims.csv`. These are retained for qualitative omission analysis; they are NOT included in the narrowly scoped claim-level precision/recall until a separate ontology and matching process is defined.
6. Enter `annotator_id` (use an anonymous consistent identifier such as A1). Do not overwrite predicted outputs; these were deliberately not provided.
7. Return the filled `gold_annotations.csv` (and `additional_security_claims.csv`, if used). A separate evaluation phase will unblind the predictions and compute results.

**Scope:** This is a rule-based source-code claim-projection check of all distinct security-relevant functions emitted by the existing Django pilot (not a test of unrestricted LLM extraction from all lifecycle artifact types). It is NOT an independent detector-accuracy evaluation.

**Validity rules:** An unsupported or unverified provenance line is not automatically evidence of a correct claim. Unresolved items are not counted as correct. The report gives denominators and separately discloses exclusions; no numerical accuracy is reported until the reference labels are actually supplied.

**Caution:** The code excerpts show individual method bodies; they may not suffice to determine the complete security consequences of inherited methods, external helpers, or multi-module interactions. Use `UNRESOLVED` rather than guessing.
""", encoding="utf-8")

    # This mapping deliberately stays OUTSIDE the blind annotation packet.
    mapping = {"schema_version": "1.0", "audit_zip_sha256": _sha256(audit_zip.read_bytes()),
               "n_raw_code_claim_observations": len(code), "n_excluded_derived_claim_records": len(other),
               "n_unique_source_units": len(records),
               "records": [{**{k: v for k, v in r.items() if k != "excerpt"},
                            "occurrences": occurrences[uid], "n_observations": len(occurrences[uid])}
                           for uid, r in sorted(records.items())],
               "methodological_scope": "9 source-code subjects; deduplicated full method/template excerpts; not full heterogeneous semantic projection"}
    (out / "PRIVATE_UNBLIND_MAPPING.json").write_text(json.dumps(mapping, indent=2), encoding="utf-8")
    (out / "PACKET_README.txt").write_text(
        "Give ONLY BLIND_ANNOTATION_PACKET.zip to the annotator. Keep PRIVATE_UNBLIND_MAPPING.json "
        "separate until gold annotations are completed.\n", encoding="utf-8")
    zip_path = out / "S3DT_Django_Blind_Claim_Annotation_Packet.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zz:
        for p in sorted(packet.rglob("*")):
            if p.is_file():
                zz.write(p, arcname=str(p.relative_to(packet)))
    return {"blind_packet": str(zip_path), "private_mapping": str(out / "PRIVATE_UNBLIND_MAPPING.json"),
            "n_unique_source_units": len(records), "n_raw_code_claim_observations": len(code),
            "n_excluded_derived_claim_records": len(other), "n_reference_rows": len(gold)}


def _parse_lines(value: str):
    value = str(value or "").strip()
    if not value:
        return set()
    ans = set()
    for part in value.split(","):
        part = part.strip()
        m = re.fullmatch(r"(\d+)(?:\s*[-–]\s*(\d+))?", part)
        if not m:
            raise ValueError(f"Invalid gold_support_lines entry: {value!r}; use 112-118 or 112,116")
        a = int(m.group(1)); b = int(m.group(2) or a)
        if a > b or b - a > 2000:
            raise ValueError(f"Invalid line range: {part}")
        ans.update(range(a,b+1))
    return ans


def _confidence_exact(successes: int, trials: int, alpha=0.05):
    if trials <= 0:
        return None, None
    try:
        from scipy.stats import beta
        lower = 0.0 if successes == 0 else float(beta.ppf(alpha / 2, successes, trials - successes + 1))
        upper = 1.0 if successes == trials else float(beta.ppf(1 - alpha / 2, successes + 1, trials - successes))
        return lower, upper
    except ImportError:
        # Wilson approximation, explicitly signaled in the output.
        import math
        z = 1.959963984540054
        p = successes/trials
        den = 1 + z*z/trials
        ctr = (p + z*z/(2*trials))/den
        half = z*math.sqrt(p*(1-p)/trials+z*z/(4*trials*trials))/den
        return max(0,ctr-half), min(1,ctr+half)


def _read_gold(source: str | Path):
    source = Path(source)
    if source.is_dir():
        file = source / "gold_annotations.csv"
        if not file.exists():
            file = source / "BLIND_ANNOTATION_PACKET" / "gold_annotations.csv"
        with file.open(encoding="utf-8-sig", newline="") as f:
            return list(csv.DictReader(f))
    if source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as zz:
            names = [n for n in zz.namelist() if n.endswith("gold_annotations.csv")]
            if len(names) != 1:
                raise ValueError(f"Expected one gold_annotations.csv in {source}, found {names}")
            with zz.open(names[0]) as f:
                return list(csv.DictReader(io.TextIOWrapper(f,encoding="utf-8-sig",newline="")))
    with source.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def score_annotations(mapping_path: str | Path, gold_file: str | Path, out_dir: str | Path):
    """Stage 2: compare completed independent reference claims with original predictions."""
    mapping = json.loads(Path(mapping_path).read_text(encoding="utf-8"))
    gold = _read_gold(gold_file)
    refs = mapping["records"]
    if len(gold) != len(refs):
        raise ValueError(f"Expected {len(refs)} reference rows; received {len(gold)}")
    by_id = {r["unit_id"]: r for r in gold}
    if len(by_id) != len(gold):
        raise ValueError("Duplicate unit_id in independent annotation file")
    pending = []
    unresolved = []
    mistakes = []
    decisions = []
    for model in refs:
        uid = model["unit_id"]
        g = by_id.get(uid)
        if not g:
            mistakes.append(f"Missing unit {uid}")
            continue
        if g.get("subject", "") != model["subject"]:
            mistakes.append(f"Subject tampered for {uid}")
        value = str(g.get("gold_object") or "").strip()
        if value == "":
            pending.append(uid)
            continue
        if value == "UNRESOLVED":
            if not str(g.get("gold_notes") or "").strip():
                mistakes.append(f"UNRESOLVED requires explanation in gold_notes: {uid}")
            unresolved.append(uid)
            continue
        if value not in REFERENCE_OPTIONS[model["subject"]]:
            mistakes.append(f"Unknown reference label for {uid}: {value}")
            continue
        if value == "OTHER" and not str(g.get("gold_notes") or "").strip():
            mistakes.append(f"OTHER requires gold_notes explaining the claim: {uid}")
        if not (g.get("annotator_id") or "").strip():
            mistakes.append(f"Missing independent annotator ID: {uid}")
        try:
            goldlines = _parse_lines(g.get("gold_support_lines", ""))
            if goldlines and (min(goldlines) < model["line_start"] or max(goldlines) > model["line_end"]):
                mistakes.append(f"Source line outside excerpt for {uid}")
        except ValueError as exc:
            mistakes.append(f"{uid}: {exc}"); goldlines = set()
        if not goldlines and value != "NONE":
            mistakes.append(f"Supported gold claim needs source line(s): {uid}")
        predicted_line = None
        match = re.search(r":(\d+)$", model["evidence"])
        if match:
            predicted_line = int(match.group(1))
        tp = int(value == model["predicted_object"])
        fp = int(not tp)
        fn = int(value not in ("NONE", model["predicted_object"]))
        decisions.append({
            "unit_id": uid, "policy_id": model["policy_id"], "subject": model["subject"],
            "predicted_object": model["predicted_object"], "gold_object": value,
            "TP": tp, "FP": fp, "FN": fn, "n_repeated_observations_excluded": model["n_observations"] - 1,
            "predicted_anchor_line": predicted_line if predicted_line else "",
            "predicted_anchor_in_gold_lines": ("YES" if predicted_line in goldlines else "NO") if goldlines and predicted_line else "NOT_SCORABLE",
            "annotator_id": g.get("annotator_id", ""), "notes": g.get("gold_notes", ""),
        })
    if pending or mistakes:
        raise ValueError(f"ANNOTATIONS NOT READY: {len(pending)} blank unit(s), {len(mistakes)} invalid unit(s). "
                         f"Pending: {pending[:8]}. Problems: {mistakes[:8]}. "
                         "No accuracy metrics were produced. Complete/adjudicate the references first.")
    if len(decisions) + len(unresolved) != len(refs):
        raise AssertionError("Internal incomplete score")
    if not decisions:
        raise ValueError("No independently resolved gold annotations; no metrics are available")
    output = Path(out_dir)
    output.mkdir(parents=True, exist_ok=True)
    cols = list(decisions[0])
    _write_csv(output / "per_unique_source_claim_comparisons.csv", decisions, cols)
    summaries = []
    for policy in ["ALL"] + sorted(POLICY_AREAS):
        sub = decisions if policy == "ALL" else [r for r in decisions if r["policy_id"] == policy]
        tp=sum(r["TP"] for r in sub); fp=sum(r["FP"] for r in sub); fn=sum(r["FN"] for r in sub)
        precision=tp/(tp+fp) if tp+fp else None
        recall=tp/(tp+fn) if tp+fn else None
        f1=2*precision*recall/(precision+recall) if precision is not None and recall is not None and precision+recall else None
        pl,pu=_confidence_exact(tp,tp+fp); rl,ru=_confidence_exact(tp,tp+fn)
        summaries.append({"policy_id":policy,"n_unique_source_units":len(sub),"TP":tp,"FP":fp,"FN":fn,
                          "precision":precision,"recall":recall,"f1":f1,
                          "precision_95CI_lower":pl,"precision_95CI_upper":pu,
                          "recall_95CI_lower":rl,"recall_95CI_upper":ru})
    _write_csv(output / "projection_precision_recall_f1.csv", summaries, list(summaries[0]))
    provenance_scored = [r for r in decisions if r["predicted_anchor_in_gold_lines"] != "NOT_SCORABLE"]
    report = {
        "scope":"historical Django code-behaviour claims in predefined 9-subject ontology",
        "n_unique_source_excerpts": len(refs),
        "n_scored_resolved_excerpts": len(decisions),
        "n_unresolved_excerpts_excluded": len(unresolved),
        "unresolved_unit_ids": unresolved,
        "n_raw_code_claim_observations_not_independent":mapping["n_raw_code_claim_observations"],
        "n_derived_activity_or_verdict_claims_excluded":mapping["n_excluded_derived_claim_records"],
        "gold_annotation_source": "human-reviewed independent reference CSV; not generated by S3DT",
        "aggregate":summaries[0],
        "provenance_alignment":{
            "n_scored":len(provenance_scored),
            "n_matching":sum(r["predicted_anchor_in_gold_lines"]=="YES" for r in provenance_scored),
            "meaning":"exact reported evidence line falls in independently cited gold support lines (not proof of full call-chain correctness)"},
        "statistical_caveat":"95% Clopper-Pearson per-claim intervals (or Wilson if scipy unavailable); repeated code deduplicated; explicitly unresolved excerpts excluded and enumerated; source excerpts within one repository remain correlated; NOT independent incidents or cross-project generalisation",
        "limitations":["restricted to predeclared Django code subjects and single best behaviour claim per excerpt",
                       "cannot estimate general LLM claim extraction performance or unrestricted recall",
                       "OTHER gold labels are treated as missed canonical claims; review separate notes",
                       "does not evaluate downstream error propagation (Stage B)"],
    }
    (output / "projection_validation_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    (output / "EVALUATION_SCOPE_AND_LIMITATIONS.md").write_text(
        "# Projection validation scope and interpretation\n\n"
        "This program does not retrain, tune, or rerun S3DT. Predicted code claims from the pinned Django evaluation "
        "are compared only AFTER independent source review. All 135 repeated code-claim observations are collapsed "
        "to their distinct code excerpts. Claims emitted by the SIC outcome stage (`risk_stage`, `activity`) "
        "are excluded. CIs are descriptive and do not establish cross-project generalisation.\n\n"
        "The ontology is restricted to nine named code subjects. Thus, the reported recall is **within the predefined "
        "extraction scope**, not recall of all possible security claims in a software project. Additional notes "
        "are not included in F1 until independently reconciled under a documented wider ontology.\n", encoding="utf-8")
    result_zip = output / "S3DT_Django_StageA_Projection_Validation_Results.zip"
    with zipfile.ZipFile(result_zip,"w",zipfile.ZIP_DEFLATED) as z:
        for file in sorted(output.iterdir()):
            if file.is_file() and file != result_zip:
                z.write(file,arcname=file.name)
    return {"summary": report, "results_zip": str(result_zip)}



## 2. Generate blind source-annotation packet

Run the following cell once. In Colab it requests the complete `TDSC_Broader_Quantitative_Audit(2).zip` if no matching audit is uploaded yet. It creates:

- `S3DT_Django_Blind_Claim_Annotation_Packet.zip` — **share this only** with someone who has not seen the S³DT predictions.
- `PRIVATE_UNBLIND_MAPPING.json` — remains in Colab, **do not share with the annotator**.

You can also directly use the annotation packet accompanying this notebook; its original 20 source units were generated from the supplied Django ZIP.

In [ ]:
from pathlib import Path
import json

possible = list(Path.cwd().glob("TDSC_Broader_Quantitative_Audit*.zip"))
if Path('/mnt/data').exists():
    possible += list(Path('/mnt/data').glob("TDSC_Broader_Quantitative_Audit*.zip"))
possible = [p for p in possible if p.stat().st_size > 1_000_000]

if not possible:
    from google.colab import files
    print("Upload the COMPLETE TDSC_Broader_Quantitative_Audit(2).zip:")
    uploaded = files.upload()
    possible = [Path(n) for n in uploaded if "TDSC_Broader_Quantitative_Audit" in n and n.endswith('.zip')]
if not possible:
    raise FileNotFoundError("A complete Django quantitative-audit ZIP is required.")

AUDIT_ZIP = max(possible, key=lambda p: p.stat().st_mtime)
STAGE_A_DIR = Path.cwd() / 'S3DT_Django_StageA'
packet_result = build_packet(AUDIT_ZIP, STAGE_A_DIR)
print(json.dumps(packet_result, indent=2))
print("IMPORTANT: Only distribute the BLIND annotation ZIP, never PRIVATE_UNBLIND_MAPPING.json")

try:
    from google.colab import files
    files.download(packet_result['blind_packet'])
except ImportError:
    print('Packet ready at:', packet_result['blind_packet'])

## 3. Independent annotation — offline step

1. Give the downloaded **blind packet ZIP** to an independent reviewer familiar with Python/Django security requirements. They should not see the S³DT outputs or the hidden prediction mapping.
2. They open each of the `source_excerpts/*.txt` files and complete the 20 rows in `gold_annotations.csv` using `REFERENCE_ONTOLOGY.csv`. They must enter a justified `gold_object`, source supporting lines, and anonymous `annotator_id`.
3. They can enter `UNRESOLVED` with reasons when the method body is insufficient to determine the fact. The scoring report **excludes and enumerates these** rather than assuming they are correct.
4. `additional_security_claims.csv` captures other relevant claims not in the nine-subject scope. These are reported separately (not silently added to the in-scope F1 denominator).
5. Ideally, obtain independent annotations from two people and adjudicate differences **before** unblinding S³DT predictions. The scoring cell expects an adjudicated reference if two annotators were used.
6. Upload the filled CSV into Colab, name it **`gold_annotations_COMPLETED.csv`**, and run Section 4.

**Do not fill gold labels by copying the system’s predicted labels.** That would reproduce the circular evaluation concern raised by Reviewer 2.

## 4. Compare predictions only after gold annotations are complete

This optional cell will **skip safely** on the first notebook run (no annotations are available yet). When a completed `gold_annotations_COMPLETED.csv` is present in the Colab Files panel, execute this cell. No additional Django detection run is needed.

In [ ]:
from pathlib import Path
import json

COMPLETED_GOLD = Path.cwd() / 'gold_annotations_COMPLETED.csv'
STAGE_A_DIR = Path.cwd() / 'S3DT_Django_StageA'
PRIVATE_MAPPING = STAGE_A_DIR / 'PRIVATE_UNBLIND_MAPPING.json'

if not COMPLETED_GOLD.exists():
    print('Stage A extraction and blind packet are ready.')
    print('No completed independent gold annotations found. This is expected for the first run.')
    print('After independent review, upload the filled CSV as gold_annotations_COMPLETED.csv and rerun this cell.')
elif not PRIVATE_MAPPING.exists():
    print('The private mapping is missing; regenerate the blind packet in Section 2 using the same audit ZIP.')
else:
    validated = score_annotations(PRIVATE_MAPPING, COMPLETED_GOLD, STAGE_A_DIR / 'validated_results')
    print(json.dumps(validated['summary'], indent=2))
    try:
        from google.colab import files
        files.download(validated['results_zip'])
    except ImportError:
        print('Validated results ZIP:', validated['results_zip'])

## 5. Interpretation for TDSC

Report the **actual independent-annotation** results only after the above scorer runs. Distinguish (a) in-scope exact claim precision/recall/F1 on deduplicated source methods, (b) source-line provenance alignment, (c) annotation exclusions and omitted claims, and (d) limitations of nine predeclared Django subjects.

The Django application case study **does not need a new implementation section** for Stage A. Include a small projection-validation paragraph/table in the empirical evaluation and put this notebook, blind annotation protocol, and adjudicated records in supplementary material or GitHub. Stage B downstream error propagation is a separate follow-on; Stage A by itself does not establish it.